# Embeddings and Vector Store

This notebook documents the embedding and vector-store stage of the biomedical RAG pipeline.

The reusable implementation lives in `src/vector_store.py`. This notebook is for inspecting and validating the design decisions and results.

## 1. Embedding model

We use `all-MiniLM-L6-v2` from Sentence Transformers as the initial embedding model.

The model converts each text chunk into a dense vector. These vectors will be stored in Qdrant and used for similarity-based retrieval.

## 2. Vector database

We use Qdrant as the vector store.

During development, Qdrant runs in local embedded mode and persists its data under `data/qdrant`. This avoids introducing Docker before the core retrieval pipeline is working.

The vector collection uses cosine similarity.

## 3. Load the generated chunks

In [1]:
import json
from pathlib import Path

chunks_path = Path("../data/pubmedqa_chunks.jsonl")

with chunks_path.open() as f:
    chunks = [json.loads(line) for line in f]

print(f"Chunks: {len(chunks)}")
print(chunks[0].keys())

Chunks: 979
dict_keys(['text', 'pubid', 'question', 'long_answer', 'final_decision', 'chunk_index', 'chunk_word_count'])


## 4. Generate embeddings

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    [chunk["text"] for chunk in chunks],
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("Embedding shape:", embeddings.shape)

/Users/lsolis/Projects/biomedical-rag-evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 31/31 [00:04<00:00,  7.48it/s]

Embedding shape: (979, 384)


## 5. Engineering rationale

The embedding model is deliberately kept replaceable. The project should eventually compare retrieval performance rather than assuming that one embedding model is optimal.

Metadata such as `pubid`, question, answer, decision label, and chunk index is stored alongside each vector. This allows retrieved chunks to be traced back to their source and supports later retrieval evaluation.

The local Qdrant database is an implementation choice for development. The project can later run Qdrant through Docker as part of the reproducible deployment setup.

## 6. Verification

After running `python ../src/vector_store.py`, the expected result is:

- all generated chunks loaded;
- embeddings generated successfully;
- a Qdrant collection named `pubmedqa` created;
- the number of stored vectors matching the number of chunks.

The next stage is retrieval: given a PubMedQA question, embed the question, search Qdrant, and inspect whether the relevant research context is returned.